In [1]:
import pandas as pd
import numpy as np

In [10]:
data = pd.read_csv('../data/drug.csv')
data

In [10]:
class_att = data.columns[-1]
class_att

In [ ]:
classes = data[class_att].unique()
classes

In [5]:
model = {}
model['_classes'] = classes
model

{'_classes': array(['drugY', 'drugC', 'drugX', 'drugA', 'drugB'], dtype=object)}

In [3]:
class_att = data.columns[-1]
class_att

'Drug'

In [7]:
apriori = data[class_att].value_counts()
apriori

Drug
drugY    91
drugX    54
drugA    23
drugC    16
drugB    16
Name: count, dtype: int64

In [11]:
smoothing = 1
apriori = (data[class_att].value_counts() + smoothing) / (len(data) + smoothing * len(classes))
apriori = apriori.replace(0, 1e-9)
apriori

Drug
drugY    0.448780
drugX    0.268293
drugA    0.117073
drugC    0.082927
drugB    0.082927
Name: count, dtype: float64

In [12]:
log_apriori = np.log(apriori)
log_apriori

Drug
drugY   -0.801221
drugX   -1.315677
drugA   -2.144956
drugC   -2.489797
drugB   -2.489797
Name: count, dtype: float64

In [13]:
features = data.drop(columns=[class_att])
features.columns

Index(['Age', 'Sex', 'BP', 'Cholesterol', 'Na', 'K'], dtype='object')

In [15]:
col = 'Age'
pd.api.types.is_numeric_dtype(data[col])

True

In [16]:
stats = data.groupby(class_att)[col].agg(['mean', 'std'])
stats

,mean,std
Drug,,
drugA,35.869565,9.696786
drugB,62.500000,7.127412
drugC,42.500000,16.725230
drugX,44.018519,16.435685
drugY,43.747253,17.031731


In [17]:
stats['std'] = stats['std'].replace(0, 1e-9)
stats

,mean,std
Drug,,
drugA,35.869565,9.696786
drugB,62.500000,7.127412
drugC,42.500000,16.725230
drugX,44.018519,16.435685
drugY,43.747253,17.031731


In [18]:
col = 'Sex'
pd.api.types.is_numeric_dtype(data[col])

False

In [19]:
mat = pd.crosstab(data[col], data[class_att])
mat

Drug,drugA,drugB,drugC,drugX,drugY
Sex,,,,,
F,9,6,7,27,47
M,14,10,9,27,44


In [20]:
n_values = mat.shape[0]
n_values

2

In [21]:
smoothing = 1.0
mat_smooth = (mat + smoothing) / (mat.sum(axis=0) + smoothing * n_values)
mat_smooth

Drug,drugA,drugB,drugC,drugX,drugY
Sex,,,,,
F,0.4,0.388889,0.444444,0.5,0.516129
M,0.6,0.611111,0.555556,0.5,0.483871


In [22]:
np.log(mat_smooth)

Drug,drugA,drugB,drugC,drugX,drugY
Sex,,,,,
F,-0.916291,-0.944462,-0.810930,-0.693147,-0.661398
M,-0.510826,-0.492476,-0.587787,-0.693147,-0.725937


In [34]:
for cls in model['_classes']:
    log_p = model['_apriori'][cls]
    print(log_p)

-0.7874578600311866
-2.5257286443082556
-1.3093333199837622
-2.162823150618887
-2.5257286443082556


In [23]:
def is_numeric(data, col):
    return pd.api.types.is_numeric_dtype(data[col])

In [24]:
def learn(data, class_att, smoothing=1.0):
    model = {}
    classes = data[class_att].unique()
    model['_classes'] = classes

    apriori = data[class_att].value_counts()
    apriori = apriori / apriori.sum()
    model['_apriori'] = np.log(apriori)

    for col in data.drop(columns=[class_att]).columns:
        if is_numeric(data, col):
            stats = data.groupby(class_att)[col].agg(['mean', 'std'])
            stats['std'] = stats['std'].replace(0, 1e-9)
            model[col] = ('gaussian', stats)
        else:
            mat = pd.crosstab(data[col], data[class_att])
            n_values = mat.shape[0]
            mat = (mat + smoothing) / (mat.sum(axis=0) + smoothing * n_values)
            model[col] = ('categorical', np.log(mat))

    return model


In [25]:
def predict(model, instance):
    log_probs = {}

    for cls in model['_classes']:
        log_p = model['_apriori'][cls]

        for col, value in instance.items():
            if col not in model:
                continue

            attr_type, attr_data = model[col]

            if attr_type == 'gaussian':
                mean = attr_data.loc[cls, 'mean']
                std  = attr_data.loc[cls, 'std']
                log_p += -0.5 * np.log(2 * np.pi * std**2) - (value - mean)**2 / (2 * std**2)
            else:
                if value in attr_data.index:
                    log_p += attr_data.loc[value, cls]
                else:
                    log_p += np.log(1e-9)

        log_probs[cls] = log_p

    prediction = max(log_probs, key=log_probs.get)
    return prediction, log_probs


In [26]:
def predict_batch(model, data):
    results = data.copy()
    predictions = []
    log_prob_cols = {cls: [] for cls in model['_classes']}

    for i in range(len(data)):
        pred, log_probs = predict(model, data.iloc[i])
        predictions.append(pred)
        for cls in model['_classes']:
            log_prob_cols[cls].append(log_probs[cls])

    results['prediction'] = predictions
    for cls in model['_classes']:
        results[f'log_P(class={cls})'] = log_prob_cols[cls]

    return results


In [27]:
model = learn(data, class_att, smoothing=1.0)
model

{'_classes': array(['drugY', 'drugC', 'drugX', 'drugA', 'drugB'], dtype=object),
 '_apriori': Drug
 drugY   -0.787458
 drugX   -1.309333
 drugA   -2.162823
 drugC   -2.525729
 drugB   -2.525729
 Name: count, dtype: float64,
 'Age': ('gaussian',
              mean        std
  Drug                       
  drugA  35.869565   9.696786
  drugB  62.500000   7.127412
  drugC  42.500000  16.725230
  drugX  44.018519  16.435685
  drugY  43.747253  17.031731),
 'Sex': ('categorical',
  Drug     drugA     drugB     drugC     drugX     drugY
  Sex                                                   
  F    -0.916291 -0.944462 -0.810930 -0.693147 -0.661398
  M    -0.510826 -0.492476 -0.587787 -0.693147 -0.725937),
 'BP': ('categorical',
  Drug       drugA     drugB     drugC     drugX     drugY
  BP                                                      
  HIGH   -0.080043 -0.111226 -2.944439 -4.043051 -0.879733
  LOW    -3.258097 -2.944439 -0.111226 -1.098612 -1.109308
  NORMAL -3.258097 -2.944439 -

In [41]:
novi_pacijent = {
        'Age': 37,
        'Sex': 'M',
        'BP': 'LOW',
        'Cholesterol': 'NORMAL',
        'NaK': 15.5
    }

predikcija, verovatnoce = predict(model, novi_pacijent)
print(f"predikcija leka za jednog: {predikcija}")
print(f"verovatnoce lekova za jednog: {verovatnoce}")

predikcija leka za jednog: drugY
verovatnoce lekova za jednog: {'drugY': np.float64(-7.1811265941713796), 'drugC': np.float64(-9.90503897139201), 'drugX': np.float64(-7.380666811462092), 'drugA': np.float64(-9.863242809108927), 'drugB': np.float64(-15.938776424826177)}


In [28]:
results = predict_batch(model, data.drop(columns=[class_att]))
results

,Age,Sex,BP,Cholesterol,Na,K,prediction,log_P(class=drugY),log_P(class=drugC),log_P(class=drugX),log_P(class=drugA),log_P(class=drugB)
0,23,F,HIGH,HIGH,0.792535,0.031258,drugY,-2.759668,-11.526426,-11.284787,-7.214971,-24.058546
1,47,M,LOW,HIGH,0.739309,0.056468,drugC,-4.553733,-2.631563,-3.457165,-6.035167,-6.951386
2,47,M,LOW,HIGH,0.697269,0.068944,drugC,-8.163608,-2.261322,-3.232367,-5.954343,-6.789631
3,28,F,NORMAL,HIGH,0.563682,0.072289,drugX,-10.977190,-6.297090,-3.449264,-6.521805,-18.613975
4,61,F,LOW,HIGH,0.559294,0.030998,drugY,-3.711198,-8.699066,-7.685770,-12.727462,-13.300456
...,...,...,...,...,...,...,...,...,...,...,...,...
195,56,F,LOW,HIGH,0.848774,0.073380,drugC,-10.462111,-4.162176,-5.194374,-9.241539,-6.593523
196,16,M,LOW,HIGH,0.743021,0.061886,drugC,-7.220776,-3.592562,-4.771726,-7.383827,-25.548313
197,52,M,NORMAL,HIGH,0.549945,0.055581,drugX,-5.915032,-6.107825,-3.049350,-7.017546,-7.889536
198,23,M,NORMAL,NORMAL,0.784520,0.055959,drugX,-5.589174,-9.279267,-3.494373,-6.649363,-20.199650


In [29]:
correct = (results['prediction'] == data[class_att]).sum()
f"Tacnost: {correct / len(data) * 100:.2f}%"

'Tacnost: 91.50%'